# Lab 2 - Two kinds of noise, and which one is actually hurting you

Exercise 14 asked you to quantify the shots a real machine produces that the
theory forbids. This lab asks the next question: **where did they come from?**

There are two answers, they behave completely differently, and the intuition most
people arrive with picks the wrong one.

Nothing here is graded, and nothing leaves the laptop. Every number below is
measured by running something, and every cell is rerunnable with different
settings.

In [ ]:
import warnings

from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, ReadoutError, depolarizing_error
from qiskit_ibm_runtime.fake_provider import FakeManilaV2

warnings.filterwarnings("ignore")

device = FakeManilaV2()
SHOTS = 8192

print(device.name, "- a 5 qubit device wired in a line: 0-1-2-3-4")

## Failure one: the answer is right, the reading is wrong

**Readout error** happens at the very end. The qubit is in the right state, and
the instrument that looks at it says the other thing.

Isolate it: no gates at all except one `x` to prepare `|1>`, and a noise model
that is nothing but a faulty detector.

In [ ]:
readout = NoiseModel()
# Deliberately lopsided: 5% of true 0s read as 1, but 10% of true 1s read as 0.
readout.add_all_qubit_readout_error(ReadoutError([[0.95, 0.05], [0.10, 0.90]]))
faulty_reader = AerSimulator(noise_model=readout)

for prepare_one in (False, True):
    qc = QuantumCircuit(1, 1)
    if prepare_one:
        qc.x(0)
    qc.measure(0, 0)
    counts = (
        faulty_reader.run(transpile(qc, faulty_reader), shots=20000, seed_simulator=2)
        .result()
        .get_counts()
    )
    prepared = "1" if prepare_one else "0"
    wrong = counts.get("1" if prepared == "0" else "0", 0)
    print(f"prepared |{prepared}>  ->  {dict(sorted(counts.items()))}   wrong {wrong / 20000:.1%}")

Two things to take from that.

It is **asymmetric**. Reading a 1 as a 0 is twice as likely as the reverse, which
is physically reasonable: an excited qubit can decay while the measurement is
happening, and a ground-state one has nowhere to fall to. So a noisy histogram is
usually biased toward `0`, not blurred evenly.

And it does not care what your circuit did. One gate or a thousand, the reading
is wrong just as often. That is the property that separates it from the other
kind.

## Failure two: the answer itself drifts

**Gate error** happens every time a gate runs, so it piles up. Here is the same
qubit flipped over and over. After an even number of `x` gates the answer must be
`0`, after an odd number `1`.

In [ ]:
gates = NoiseModel()
gates.add_all_qubit_quantum_error(depolarizing_error(0.02, 1), ["x"])
shaky_gates = AerSimulator(noise_model=gates)

print(f"{'x gates':>8}{'correct':>10}")
for depth in (0, 2, 10, 50, 200):
    qc = QuantumCircuit(1, 1)
    for _ in range(depth):
        qc.x(0)
    qc.measure(0, 0)
    # optimization_level=0, or the transpiler cancels the gates in pairs and the
    # whole point of the experiment disappears.
    counts = (
        shaky_gates.run(
            transpile(qc, shaky_gates, optimization_level=0), shots=SHOTS, seed_simulator=3
        )
        .result()
        .get_counts()
    )
    expected = "0" if depth % 2 == 0 else "1"
    print(f"{depth:>8}{counts.get(expected, 0) / SHOTS:>10.4f}")

It decays toward **0.5**, not toward 0. That is worth sitting with: a fully
depolarised qubit is not wrong, it is random. Half the shots still agree with you
by luck, and a result of 0.5 carries no information at all.

Each gate is only 2% wrong here, which sounds harmless. Two hundred of them and
the qubit has forgotten everything.

## What the machine itself says

Both numbers are published. A backend's `target` carries the measured error rate
of every operation on every qubit, and this is where circuit design starts.

In [ ]:
target = device.target
print(f"{'qubit':>6}{'readout':>10}{'sx gate':>10}{'T1 (us)':>10}{'T2 (us)':>10}")
for q in range(device.num_qubits):
    props = device.qubit_properties(q)
    print(
        f"{q:>6}"
        f"{target['measure'][(q,)].error:>10.4f}"
        f"{target['sx'][(q,)].error:>10.5f}"
        f"{props.t1 * 1e6:>10.1f}"
        f"{props.t2 * 1e6:>10.1f}"
    )

pairs = sorted(target["cx"])
print("\ntwo qubit gates, which are the expensive ones:")
for pair in pairs[:4]:
    print(f"  cx{pair}: {target['cx'][pair].error:.4f}")

Read the readout column again, then the one beside it. Readout error runs from
1.4% to 9.6% across these five qubits. The `sx` gate runs from 0.016% to 0.075%.
That is between fifty and two hundred times worse, depending which qubit you land
on, and it is on the operation people think about least.

And the qubits are not equal. Qubit 2 reads back wrong nearly **10%** of the time,
four to seven times worse than its neighbours. On a device wired as a line, that
one bad qubit sits in the middle of everything.

## Benchmarking: how bad is it really?

A GHZ state is the standard stress test. All qubits entangled, and only two
outcomes are legal: all zeros or all ones. Anything else is noise, so counting the
legal shots gives a single number to compare against.

The third column is the interesting one. It runs the same circuit with a noise
model containing **nothing but the device's own readout errors**, no gate errors
at all.

In [ ]:
def ghz(n):
    qc = QuantumCircuit(n)
    qc.h(0)
    for q in range(n - 1):
        qc.cx(q, q + 1)
    qc.measure_all()
    return qc


def legal_fraction(simulator, circuit, n):
    isa = transpile(circuit, simulator, optimization_level=1, seed_transpiler=7)
    counts = simulator.run(isa, shots=SHOTS, seed_simulator=5).result().get_counts()
    return (counts.get("0" * n, 0) + counts.get("1" * n, 0)) / SHOTS


everything = NoiseModel.from_backend(device)

readout_only = NoiseModel()
for q in range(device.num_qubits):
    rate = target["measure"][(q,)].error
    # A symmetric stand-in built from the one number the device publishes.
    readout_only.add_readout_error(ReadoutError([[1 - rate, rate], [rate, 1 - rate]]), [q])

print(f"{'qubits':>7}{'full noise':>12}{'readout only':>14}{'noiseless':>11}")
for n in (2, 3, 4, 5):
    row = [
        legal_fraction(AerSimulator(noise_model=everything), ghz(n), n),
        legal_fraction(AerSimulator(noise_model=readout_only), ghz(n), n),
        legal_fraction(AerSimulator(), ghz(n), n),
    ]
    print(f"{n:>7}{row[0]:>12.4f}{row[1]:>14.4f}{row[2]:>11.4f}")

That is the result worth carrying out of this lab.

**Readout error alone accounts for almost the whole gap.** At five qubits the full
model gives about 0.81 and readout on its own gives about 0.82: every gate in the
circuit, all the entangling ones included, costs barely a point.

Anyone tuning gate fidelity on this device, on circuits this shallow, is polishing
the wrong thing.

That is not a universal law. It is true here because the circuits are shallow and
this device's readout is poor relative to its gates. Deepen the circuit and the
balance flips, which is exactly what the second experiment above showed.

## So do something about it

The cheapest fix needs no new theory. Some qubits read back better than others, so
ask for those.

In [ ]:
noisy = AerSimulator.from_backend(device)
bell = QuantumCircuit(2)
bell.h(0)
bell.cx(0, 1)
bell.measure_all()

print(f"{'physical qubits':>16}{'measured':>11}{'readout alone predicts':>25}")
for layout in ([0, 1], [1, 2], [3, 4]):
    isa = transpile(bell, noisy, initial_layout=layout, optimization_level=1, seed_transpiler=7)
    counts = noisy.run(isa, shots=SHOTS, seed_simulator=11).result().get_counts()
    measured = (counts.get("00", 0) + counts.get("11", 0)) / SHOTS

    predicted = 1.0
    for q in layout:
        predicted *= 1 - target["measure"][(q,)].error

    print(f"{str(layout):>16}{measured:>11.4f}{predicted:>25.4f}")

Same circuit, same device, same shot count. Choosing qubits 3 and 4 instead of 1
and 2 takes the result from about 0.88 to about 0.97, and the only thing that
changed was which physical qubits the transpiler was told to use.

Look at the right-hand column too. Multiplying the two published readout rates
predicts the measured value to better than half a percent, three times running.
The model is not a metaphor here, it is arithmetic that lands.

## Where to go next

- Set `optimization_level=3` in the GHZ cell and see how much the transpiler can
  claw back on its own.
- Raise `depolarizing_error(0.02, 1)` to `0.1` and find the depth where the answer
  becomes worthless.
- Swap `FakeManilaV2` for another entry in `qiskit_ibm_runtime.fake_provider` and
  rerun. A newer device changes which of the two failures wins. One thing to fix
  when you do: the table above asks the target for `cx`, and the Heron and Eagle
  families use `ecr` or `cz` instead, so that one line needs the name the device
  actually has.

Exercise 14 is the companion to this: given a noisy histogram and no other
information, decide whether you are looking at a bug, at noise, or at physics.